# 01 · Load and clean

**Goal:** turn two raw exports into clean, pseudonymised tables I can query with SQL.

| Input (`data/raw/`, private) | What it is |
|---|---|
| `sales_export.csv` | Every order from the platform I sell beat licences on, since January 2020 |
| `email_list_export.csv` | Everyone currently subscribed to my email list |

| Output (private) | What it is |
|---|---|
| `data/clean/*.csv` | Four clean tables, no names or emails |
| `data/work/music_business.db` | The same four tables in a SQLite database, ready for SQL |

**Privacy rules for this notebook**

- Customer names and emails never appear in an output cell, because this notebook is published.
- Emails are replaced with a `customer_id`: a one-way hash, salted with a secret kept in `data/work/`. The same email always gets the same ID, so the two files can still be joined, but the ID can't be turned back into an email.
- No business figures are printed: no counts of customers, orders, subscribers or beats, and no revenue. Checks print as shares or pass/fail.
- Anything specific to my accounts (file layouts, account names, platform labels) lives in a private settings file, `data/work/source_config.json`, so the code itself stays generic.

In [1]:
import csv
import hashlib
import json
import secrets
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
WORK = ROOT / "data" / "work"
CLEAN = ROOT / "data" / "clean"
WORK.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

CFG = json.loads((WORK / "source_config.json").read_text())
SALES_FILE = RAW / "sales_export.csv"
EMAIL_FILE = RAW / "email_list_export.csv"
print("Settings loaded:", ", ".join(CFG))

Settings loaded: my_account, guest_marker, title_suffix, store_types, email_columns, email_sources, email_timezone, tags


## 1. Look at the sales file before loading it

A normal `pd.read_csv` fails on this file, so first I look at its *shape*: how many fields each line has. I print counts only, never the contents.

In [2]:
lines = SALES_FILE.read_text(encoding="utf-8-sig").splitlines()
rows = list(csv.reader(lines))

print("Fields per line, first 16 lines:")
print([len(r) for r in rows[:16]])

Fields per line, first 16 lines:
[1, 0, 14, 14, 15, 15, 0, 14, 0, 14, 0, 14, 14, 0, 14, 14]


**What this shows**

- Line 1 is a title, line 2 is blank, and the column names are on **line 3**.
- Orders are separated by **blank lines** (0 fields).
- Each order is a **block** of rows. Inside a block there are three kinds of row:
  1. **Order row:** has an order number, date and customer, plus the first beat and the first payment split.
  2. **Extra item row:** no order number, but a beat title. Another beat bought in the same order.
  3. **Extra split row:** only the split payment columns filled. Another person's share of the beat above it (a producer I co-produced it with).
- A few lines have **15 fields** instead of 14. That happens when a beat title contains a comma and the export didn't quote it properly, so the title spills into the next column.

## 2. Parse the blocks into rows I can work with

I walk through the file line by line, remembering which order and which item I'm inside, and tag every row with its type. The 14 columns get my own names.

In [3]:
COLUMNS = [
    "order_number", "date", "status", "store_type", "customer_name",
    "customer_email", "item_name", "list_price", "discount", "sale_price",
    "split_party", "split_type", "processing_fee", "split_amount",
]

records = []
fixed_titles = 0
order_no = 0
item_no = 0

for r in rows[3:]:                      # skip title, blank line and header
    if not r:                           # blank line between orders
        continue

    if len(r) > 14:                     # a comma inside a beat title split it
        extra = len(r) - 14
        r = r[:6] + [",".join(r[6:7 + extra])] + r[7 + extra:]
        fixed_titles += 1

    rec = dict(zip(COLUMNS, (v.strip() for v in r)))

    if rec["order_number"]:
        order_no += 1
        item_no = 1
        rec["row_type"] = "order"
    elif rec["item_name"]:
        item_no += 1
        rec["row_type"] = "extra_item"
    else:
        rec["row_type"] = "extra_split"

    rec["order_no"] = order_no
    rec["item_no"] = item_no
    records.append(rec)

raw = pd.DataFrame(records)
assert fixed_titles > 0
print("Beat titles rejoined after a stray comma: yes")
print("Row types found:", sorted(raw["row_type"].unique()))

Beat titles rejoined after a stray comma: yes
Row types found: ['extra_item', 'extra_split', 'order']


## 3. Orders table: one row per order

Order-level details only live on the order row, so I take those rows and clean each column.

**Dates** look like `Month DD, YYYY Weekday, HH:MM:SS PM EDT`. I remove the weekday, read the rest, and convert to UTC using the time zone at the end (EDT is UTC-4, EST is UTC-5), so they line up with the email list's times.

In [4]:
orders = raw[raw["row_type"] == "order"].copy()

tz = orders["date"].str.extract(r"(E[DS]T)$")[0]
local = pd.to_datetime(
    orders["date"].str.replace(r"\s\w+day,", "", regex=True).str.replace(r"\sE[DS]T$", "", regex=True),
    format="%B %d, %Y %I:%M:%S %p",
)
orders["order_time_utc"] = local + pd.to_timedelta(tz.map({"EDT": 4, "EST": 5}), unit="h")

print("Time zones found:", sorted(tz.unique()))
assert orders["order_time_utc"].notna().all()
print("Every date parsed")

Time zones found: ['EDT', 'EST']
Every date parsed


**Status** is either blank or a list of payment IDs, each marked `COMPLETED` or `PENDING`. I reduce it to one word. Blank means the export didn't record a payment status. Most of those orders have a sale price, so they look like real sales, but I keep them as `not_recorded` rather than guessing.

**Channel** says whether the order came through the platform's public marketplace or through my own storefront. **Guest checkout** means the buyer didn't use an account; it's marked in the order number.

In [5]:
def order_status(s):
    if "PENDING" in s:
        return "pending"
    if "COMPLETED" in s:
        return "completed"
    return "not_recorded"

orders["status"] = orders["status"].map(order_status)
orders["is_guest_checkout"] = orders["order_number"].str.contains(CFG["guest_marker"])
orders["channel"] = orders["store_type"].map(CFG["store_types"])
assert orders["channel"].notna().all()

share = lambda s: (s.value_counts(normalize=True) * 100).round(0).astype(int).astype(str) + "%"
print("Status\n" + share(orders["status"]).to_string(), "\n")
print("Channel\n" + share(orders["channel"]).to_string(), "\n")
print(f"Guest checkouts: {orders['is_guest_checkout'].mean():.0%} of orders")

Status
status
completed       86%
not_recorded    14%
pending          0% 

Channel
channel
marketplace    52%
own_store      47%
other           0% 

Guest checkouts: 49% of orders


## 4. Customers: replace emails with an ID

Emails are cleaned (lowercase, spaces removed) so that `Name@Gmail.com ` and `name@gmail.com` count as the same person. Then each one is hashed with a secret salt.

The salt is created once and saved in `data/work/salt.txt`, which is never published. Without it, nobody can check whether a given email is in the data.

In [6]:
salt_file = WORK / "salt.txt"
if not salt_file.exists():
    salt_file.write_text(secrets.token_hex(16))
SALT = salt_file.read_text().strip()

def normalise_email(e):
    e = str(e).strip().lower()
    return e if "@" in e else None

def customer_id(email):
    if email is None:
        return None
    return "c_" + hashlib.sha256((SALT + email).encode()).hexdigest()[:12]

orders["customer_id"] = orders["customer_email"].map(normalise_email).map(customer_id)

assert orders["customer_id"].notna().all()
print("Every order has a customer ID")

Every order has a customer ID


Each order gets a simple `order_id` (numbered by date, oldest first) in place of the platform's order number, which contains account numbers. Then I keep only the columns the analysis needs.

In [7]:
orders = orders.sort_values("order_time_utc").reset_index(drop=True)
orders["order_id"] = range(1, len(orders) + 1)
order_id_map = dict(zip(orders["order_no"], orders["order_id"]))

orders_clean = orders[[
    "order_id", "order_time_utc", "customer_id", "channel",
    "is_guest_checkout", "status",
]]
print("Orders table columns:", ", ".join(orders_clean.columns))

Orders table columns: order_id, order_time_utc, customer_id, channel, is_guest_checkout, status


## 5. Order items: one row per beat bought

Items come from order rows plus extra item rows. Beat titles need tidying: stray quote marks from the broken export, and a label the platform adds to the end of every title.

Prices are text, so I convert them to numbers. **Discount** is the amount taken off the list price; when it equals the list price, the beat was free (a 100% coupon or a free download that went through checkout).

In [8]:
items = raw[raw["row_type"].isin(["order", "extra_item"])].copy()

items["beat_title"] = (
    items["item_name"]
    .str.replace('"', "", regex=False)
    .str.replace(CFG["title_suffix"], "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

for col in ["list_price", "discount", "sale_price"]:
    items[col] = pd.to_numeric(items[col], errors="coerce")

items["is_free"] = items["sale_price"] == 0
items["order_id"] = items["order_no"].map(order_id_map)

items_clean = items[[
    "order_id", "item_no", "beat_title", "list_price", "discount", "sale_price", "is_free",
]].sort_values(["order_id", "item_no"]).reset_index(drop=True)

assert items_clean[["list_price", "discount", "sale_price"]].notna().all().all()
print("Every price converted")
print(f"Free items: {items_clean['is_free'].mean():.0%} of items")

Every price converted
Free items: 13% of items


## 6. Payment splits: who got paid what

Every item can be split between me and the producers I made it with. Each row says who, their role, the processing fee and their amount. My own account becomes `me`, and every collaborator becomes `collab_01`, `collab_02` and so on, since they're other producers and not part of this story.

**Why this matters:** the sale price is what the customer paid. What *I* earned is only my split.

In [9]:
splits = raw.copy()
splits["order_id"] = splits["order_no"].map(order_id_map)

for col in ["processing_fee", "split_amount"]:
    splits[col] = pd.to_numeric(splits[col], errors="coerce")

mine_name = CFG["my_account"]
others = [p for p in splits["split_party"].drop_duplicates() if p != mine_name]
party_map = {mine_name: "me"} | {p: f"collab_{i:02d}" for i, p in enumerate(others, 1)}
splits["party"] = splits["split_party"].map(party_map)
splits["split_role"] = splits["split_type"].str.lower()

splits_clean = splits[[
    "order_id", "item_no", "party", "split_role", "processing_fee", "split_amount",
]].sort_values(["order_id", "item_no"]).reset_index(drop=True)

mine = splits_clean[splits_clean["party"] == "me"][["order_id", "item_no"]].drop_duplicates()
assert len(mine) == len(items_clean)
print("Every item has my share recorded")
print(f"Items with at least one collaborator: {splits_clean.groupby(['order_id', 'item_no'])['party'].apply(lambda p: (p != 'me').any()).mean():.0%}")

Every item has my share recorded


Items with at least one collaborator: 68%


## 7. Email list subscribers

This file is already a proper table. Cleaning it means:

- keeping only the columns the analysis needs;
- hashing emails with the **same salt**, so subscribers and buyers can be joined on `customer_id`;
- turning the tags into true/false columns. Some tags are stored with doubled-up quotes, so I strip all quotes before matching;
- keeping **source**: `auto_signup` means the sales platform added them automatically (through a free download or a purchase), and `file_import` means I added them from a spreadsheet in 2021;
- **converting opt-in times to UTC.** The export doesn't say which time zone it uses. When I first joined the two files, most buyers appeared to subscribe exactly 4 or 5 hours before buying: the gap between US Eastern time and UTC. So the email platform appears to record US Eastern time, and I convert it like the sales dates.

In [10]:
col = CFG["email_columns"]
em = pd.read_csv(EMAIL_FILE, low_memory=False)

em["customer_id"] = em[col["email"]].map(normalise_email).map(customer_id)
em["optin_time_utc"] = (
    pd.to_datetime(em[col["optin"]])
    .dt.tz_localize(CFG["email_timezone"], ambiguous="NaT", nonexistent="shift_forward")
    .dt.tz_convert("UTC")
    .dt.tz_localize(None)
)

tags = em[col["tags"]].fillna("").str.replace('"', "", regex=False).str.replace("\\", "", regex=False)
em["tag_free_download"] = tags.str.contains(CFG["tags"]["free_download"], regex=False)
em["tag_sales"] = tags.str.contains(CFG["tags"]["sales"], regex=False)
em["source"] = em[col["source"]].map(CFG["email_sources"])
assert em["source"].notna().all()

subscribers_clean = em[[
    "customer_id", "optin_time_utc", "source", "tag_free_download", "tag_sales",
]].sort_values("optin_time_utc").reset_index(drop=True)

assert not subscribers_clean["customer_id"].duplicated().any()
print("No duplicate subscribers")
print("Source\n" + (subscribers_clean["source"].value_counts(normalize=True) * 100).round(0).astype(int).astype(str).add("%").to_string())

No duplicate subscribers
Source
source
auto_signup    76%
file_import    24%
manual          0%


## 8. Checks before saving

A few things that must be true. If any check fails, the cell stops with an error.

In [11]:
# No personal data in anything I'm about to save
for name, df in [("orders", orders_clean), ("items", items_clean), ("splits", splits_clean), ("subscribers", subscribers_clean)]:
    text = df.astype(str).apply(" ".join, axis=1)
    assert not text.str.contains("@").any(), f"email found in {name}"

# My account name is replaced in the splits table
assert not (splits_clean["party"] == mine_name).any()

# Every item and split belongs to a real order
assert items_clean["order_id"].isin(orders_clean["order_id"]).all()
assert splits_clean["order_id"].isin(orders_clean["order_id"]).all()

# One row per order, one row per subscriber
assert orders_clean["order_id"].is_unique
assert subscribers_clean["customer_id"].is_unique

buyers = set(orders_clean["customer_id"].dropna())
subs = set(subscribers_clean["customer_id"])
print("All checks passed")
print(f"Buyers who are also current subscribers: {len(buyers & subs) / len(buyers):.0%}")

All checks passed
Buyers who are also current subscribers: 45%


## 9. Save

CSV files go to `data/clean/`, and the same tables go into a SQLite database in `data/work/`. Both stay on this computer: `.gitignore` keeps everything in `data/` private, because the tables still contain my revenue.

In [12]:
tables = {
    "orders": orders_clean,
    "order_items": items_clean,
    "payment_splits": splits_clean,
    "subscribers": subscribers_clean,
}

db_path = WORK / "music_business.db"
with sqlite3.connect(db_path) as con:
    for name, df in tables.items():
        df.to_csv(CLEAN / f"{name}.csv", index=False)
        df.to_sql(name, con, if_exists="replace", index=False)

for name, df in tables.items():
    print(f"{name:15} {', '.join(df.columns)}")

orders          order_id, order_time_utc, customer_id, channel, is_guest_checkout, status
order_items     order_id, item_no, beat_title, list_price, discount, sale_price, is_free
payment_splits  order_id, item_no, party, split_role, processing_fee, split_amount
subscribers     customer_id, optin_time_utc, source, tag_free_download, tag_sales


The analysis continues in `02_sql_analysis.ipynb`, which queries `data/work/music_business.db`.